# Basic Quadrature Rules

Example 4.6: approximate
$$
I(f) = \int_0^1 \frac{1}{1+x^2}\,dx = \frac{\pi}{4}
$$
using the left endpoint, right endpoint, trapezoid, midpoint, and Simpson rules.

In [ ]:
import math

In [ ]:
def f(x):
    return 1 / (1 + x**2)


a, b = 0.0, 1.0
c = (a + b) / 2
h = b - a
exact = math.pi / 4

## Approximations, errors, and error bounds

On $[0,1]$, the derivative norms are
$$
\|f'\| = \frac{3\sqrt{3}}{8}, \qquad \|f''\| = 2, \qquad \|f^{(4)}\| = 24.
$$
Substituting these into the error formulas gives the bounds in the last column (Table 4.2).

In [ ]:
norm1 = 3 * math.sqrt(3) / 8
norm2 = 2
norm4 = 24

rules = [
    ("Left", h * f(a), norm1 * h**2 / 2),
    ("Right", h * f(b), norm1 * h**2 / 2),
    ("Trapezoid", h * (f(a) + f(b)) / 2, norm2 * h**3 / 12),
    ("Midpoint", h * f(c), norm2 * h**3 / 24),
    ("Simpson", h * (f(a) + 4 * f(c) + f(b)) / 6, norm4 * h**5 / 2880),
]

print(f"{'Rule':<10} | {'Q(f)':>14} | {'Error':>14} | {'Error bound':>14}")
print("-" * 61)
for name, value, bound in rules:
    print(f"{name:<10} | {value:14.10f} | {abs(exact - value):14.6e} | {bound:14.6e}")

## Composite quadrature

Apply a basic rule on each of $n$ equal subintervals and add the results.
We can compute the trapezoid and Simpson rules from the other three:
$$
Q_T = \frac{Q_L + Q_R}{2}, \qquad Q_S = \frac{2Q_M + Q_T}{3}.
$$

In [ ]:
def left(f, a, b, n):
    h = (b - a) / n
    return h * sum(f(a + j * h) for j in range(n))


def right(f, a, b, n):
    h = (b - a) / n
    return h * sum(f(a + j * h) for j in range(1, n + 1))


def trapezoid(f, a, b, n):
    return (left(f, a, b, n) + right(f, a, b, n)) / 2


def midpoint(f, a, b, n):
    h = (b - a) / n
    return h * sum(f(a + j * h + h / 2) for j in range(n))


def simpson(f, a, b, n):
    return (2 * midpoint(f, a, b, n) + trapezoid(f, a, b, n)) / 3

## Example 4.8

Approximate $\int_0^1 e^x\,dx = e-1$ for $n = 1, 2, 4, \ldots, 128$.
For each rule, we print the signed error $e_n = I(f) - Q_n(f)$, the ratio
$r_n = |e_{n/2}/e_n|$, and the observed order $p_n = \log_2 r_n$.
We expect orders 1 for the endpoint rules, 2 for trapezoid and midpoint, and 4 for Simpson.

In [ ]:
exact = math.e - 1

for q in [left, right, trapezoid, midpoint, simpson]:
    print(f"\n{q.__name__.capitalize()}")
    print(f"{'n':>5} | {'Error':>14} | {'Ratio':>10} | {'p':>8}")
    print("-" * 46)
    previous = None
    for k in range(8):
        n = 2**k
        error = exact - q(math.exp, 0.0, 1.0, n)
        row = f"{n:5d} | {error:14.6e}"
        if previous is not None:
            ratio = abs(previous / error)
            row += f" | {ratio:10.4f} | {math.log2(ratio):8.4f}"
        print(row)
        previous = error

## Example 4.9

Apply the trapezoid rule to $f_m(x) = x^m(1-x)^m$ on $[0,1]$, where
$$
I(f_m) = \frac{(m!)^2}{(2m+1)!}.
$$
We use exact rational arithmetic, as in the textbook, so rounding does not hide the small errors.
Each table shows $e_n = |I(f_m) - Q_n(f_m)|$ and $p_n = \log_2(e_{n/2}/e_n)$.
The observed orders approach 2, 4, 4, 6, 6, and 8 for $m = 1, \ldots, 6$.

In [ ]:
from fractions import Fraction

for m in range(1, 7):
    def f_m(x):
        return x**m * (1 - x)**m

    exact = Fraction(math.factorial(m)**2, math.factorial(2 * m + 1))
    previous = exact  # For n = 1, both endpoint values are zero.
    print(f"\nm = {m}")
    print(f"{'n':>5} | {'Error':>14} | {'p':>8}")
    print("-" * 33)
    for k in range(1, 11):
        n = 2**k
        value = trapezoid(f_m, Fraction(0), Fraction(1), n)
        error = abs(exact - value)
        p = math.log2(previous / error)
        print(f"{n:5d} | {float(error):14.6e} | {p:8.4f}")
        previous = error